# 02 CI And Mask Analysis

This notebook evaluates lower / upper CI statistics and expected masked weights. In the new reframing, these
plots diagnose **CI-induced attenuation**, not shrinkage in the paper's ML2R sense.


In [1]:
from pathlib import Path
import sys

if (Path.cwd() / 'koko_notebooks').exists():
    REPO_ROOT = Path.cwd().resolve()
else:
    REPO_ROOT = Path.cwd().resolve().parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from koko_notebooks.shrinkage_analysis.shrinkage_analysis_common import (
    OUTPUTS_DIR,
    PLOTS_DIR,
    RESULTS_DIR,
    batched_expected_masked_weight,
    batched_expected_train_mask_weight,
    collect_ci_outputs,
    component_strengths,
    discover_exp07_analysis_jsons,
    ensure_outputs_dir,
    exhaustive_binary_probe_batch,
    latest_result_per_run,
    layer_weight_metric_row,
    load_component_model_for_checkpoint,
    save_dataframe,
    save_json,
    sampled_probe_batch,
    select_consistent_replicate,
    singleton_probe_batch,
)
from koko_notebooks.shrinkage_analysis.publication_plots import (
    ARCH_COLORS,
    LAYER_COLORS,
    architecture_comparison_plot,
    heatmap,
    histogram_triptych,
    line_plot_by_group,
    line_plot_by_layer,
    multi_metric_panel_by_group,
    multi_metric_panel_by_layer,
    parse_vector_column,
    save_figure,
    setup_publication_style,
    singular_value_trajectory_plot,
)

ensure_outputs_dir()
setup_publication_style()
OUTPUTS_DIR


/root/spd_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1')

In [2]:
import pandas as pd
from tqdm.auto import tqdm

CONSISTENT_REPLICATE = 1

manifest_df = latest_result_per_run(discover_exp07_analysis_jsons())
manifest_df = select_consistent_replicate(manifest_df, CONSISTENT_REPLICATE)
manifest_df = manifest_df.sort_values(['depth', 'architecture', 'replicate', 'run_name']).reset_index(drop=True)

SELECTED_DEPTHS = [2, 3, 4, 5, 6]
SELECTED_ARCHITECTURES = ['tied', 'untied']
PROBE_TYPE = 'singleton'
INPUT_MAGNITUDE = 1.0
SAMPLED_BATCH_SIZE = 256
SAMPLED_SEED = 0
DEVICE = 'cpu'
LAYER_ORDER = list(LAYER_COLORS.keys())

selected_manifest_df = manifest_df[
    manifest_df['depth'].isin(SELECTED_DEPTHS) & manifest_df['architecture'].isin(SELECTED_ARCHITECTURES)
].copy()
selected_manifest_df[['run_name', 'depth', 'architecture', 'replicate']]


,run_name,depth,architecture,replicate
0,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1
1,exp_07_tms_5_2_2layer_untied_rep1,2,untied,1
2,exp_07_tms_5_2_3layer_tied_rep1,3,tied,1
3,exp_07_tms_5_2_3layer_untied_rep1,3,untied,1
4,exp_07_tms_5_2_4layer_tied_rep1,4,tied,1
5,exp_07_tms_5_2_4layer_untied_rep1,4,untied,1
6,exp_07_tms_5_2_5layer_tied_rep1,5,tied,1
7,exp_07_tms_5_2_5layer_untied_rep1,5,untied,1
8,exp_07_tms_5_2_6layer_tied_rep1,6,tied,1
9,exp_07_tms_5_2_6layer_untied_rep1,6,untied,1


In [3]:
def make_probe_batch(config, device: str):
    task_config = config.task_config
    if PROBE_TYPE == 'singleton':
        return singleton_probe_batch(n_features=5, input_magnitude=INPUT_MAGNITUDE, device=device)
    if PROBE_TYPE == 'exhaustive':
        return exhaustive_binary_probe_batch(n_features=5, device=device)
    return sampled_probe_batch(task_config=task_config, batch_size=SAMPLED_BATCH_SIZE, device=device, seed=SAMPLED_SEED)

ci_rows: list[dict[str, object]] = []
for _, manifest_row in selected_manifest_df.iterrows():
    spd_run_dir = Path(manifest_row['spd_run_dir'])
    for step in tqdm(manifest_row['checkpoint_steps'], desc=manifest_row['run_name']):
        component_model, _target_model, config = load_component_model_for_checkpoint(spd_run_dir=spd_run_dir, step=int(step), device=DEVICE)
        probe_batch = make_probe_batch(config=config, device=DEVICE)
        pre_sigmoid, ci_lower, ci_upper = collect_ci_outputs(component_model=component_model, batch=probe_batch, sampling=config.sampling)
        weight_deltas = component_model.calc_weight_deltas()
        for layer_name, components in component_model.components.items():
            target_weight = component_model.target_weight(layer_name).detach()
            target_fro = target_weight.norm().clamp_min(1e-12)
            raw_ratio = float((components.weight.detach().norm() / target_fro).item())
            expected_masked_weight = batched_expected_masked_weight(components=components, ci_lower=ci_lower[layer_name])
            expected_train_weight = batched_expected_train_mask_weight(components=components, ci_lower=ci_lower[layer_name], delta_weight=weight_deltas[layer_name])
            expected_mask_ratio = float((expected_masked_weight.norm(dim=(1, 2)) / target_fro).mean().item())
            expected_train_ratio = float((expected_train_weight.norm(dim=(1, 2)) / target_fro).mean().item())
            ci_rows.append(
                {
                    'run_name': manifest_row['run_name'],
                    'depth': int(manifest_row['depth']),
                    'architecture': manifest_row['architecture'],
                    'replicate': int(manifest_row['replicate']),
                    'checkpoint_step': int(step),
                    'layer_name': layer_name,
                    'probe_type': PROBE_TYPE,
                    'mean_pre_sigmoid_ci': float(pre_sigmoid[layer_name].mean().item()),
                    'mean_lower_ci': float(ci_lower[layer_name].mean().item()),
                    'mean_upper_ci': float(ci_upper[layer_name].mean().item()),
                    'median_lower_ci': float(ci_lower[layer_name].median().item()),
                    'frac_lower_ci_gt_0p9': float((ci_lower[layer_name] > 0.9).float().mean().item()),
                    'frac_lower_ci_lt_0p1': float((ci_lower[layer_name] < 0.1).float().mean().item()),
                    'raw_fro_ratio': raw_ratio,
                    'expected_mask_fro_ratio_mean': expected_mask_ratio,
                    'expected_train_mask_fro_ratio_mean': expected_train_ratio,
                    'mask_attenuation_gap': raw_ratio - expected_mask_ratio,
                    'train_mask_attenuation_gap': raw_ratio - expected_train_ratio,
                }
            )

ci_mask_df = pd.DataFrame(ci_rows)
ci_mask_df.head()


exp_07_tms_5_2_2layer_tied_rep1:   0%|          | 0/8 [00:00<?, ?it/s]

exp_07_tms_5_2_2layer_tied_rep1:  12%|█▎        | 1/8 [00:01<00:09,  1.30s/it]

exp_07_tms_5_2_2layer_tied_rep1:  38%|███▊      | 3/8 [00:01<00:01,  2.62it/s]

exp_07_tms_5_2_2layer_tied_rep1:  62%|██████▎   | 5/8 [00:01<00:00,  4.53it/s]

exp_07_tms_5_2_2layer_tied_rep1:  88%|████████▊ | 7/8 [00:01<00:00,  6.20it/s]

...

exp_07_tms_5_2_6layer_untied_rep1:  62%|██████▎   | 5/8 [00:00<00:00,  7.85it/s]

exp_07_tms_5_2_6layer_untied_rep1:  88%|████████▊ | 7/8 [00:00<00:00,  8.68it/s]

exp_07_tms_5_2_6layer_untied_rep1: 100%|██████████| 8/8 [00:01<00:00,  8.95it/s]

exp_07_tms_5_2_6layer_untied_rep1: 100%|██████████| 8/8 [00:01<00:00,  7.99it/s]

,run_name,depth,architecture,replicate,checkpoint_step,layer_name,probe_type,mean_pre_sigmoid_ci,mean_lower_ci,mean_upper_ci,median_lower_ci,frac_lower_ci_gt_0p9,frac_lower_ci_lt_0p1,raw_fro_ratio,expected_mask_fro_ratio_mean,expected_train_mask_fro_ratio_mean,mask_attenuation_gap,train_mask_attenuation_gap
0,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,linear1,singleton,-0.226405,0.050000,0.051725,0.0,0.05,0.95,1.002558,0.634712,0.633492,0.367846,0.369065
1,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,linear2,singleton,-0.846065,0.117893,0.121287,0.0,0.06,0.85,1.002558,0.772655,0.771400,0.229903,0.231158
2,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,linear1,singleton,-0.488689,0.049326,0.049978,0.0,0.05,0.95,1.000920,0.631940,0.631504,0.368980,0.369416
3,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,linear2,singleton,-1.758759,0.118355,0.120637,0.0,0.05,0.85,1.000920,0.773281,0.772828,0.227639,0.228092
4,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,15000,linear1,singleton,-0.560046,0.049281,0.049608,0.0,0.05,0.95,1.001431,0.632484,0.631799,0.368947,0.369632


In [4]:
ci_mask_csv = save_dataframe(ci_mask_df, 'csv/ci_mask_metrics.csv')
ci_mask_csv


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1/csv/ci_mask_metrics.csv')

In [5]:
ci_mean_df = (
    ci_mask_df.groupby(['depth', 'architecture', 'checkpoint_step', 'layer_name'], as_index=False)
    .agg(
        mean_lower_ci=('mean_lower_ci', 'mean'),
        mean_upper_ci=('mean_upper_ci', 'mean'),
        median_lower_ci=('median_lower_ci', 'mean'),
        frac_lower_ci_gt_0p9=('frac_lower_ci_gt_0p9', 'mean'),
        frac_lower_ci_lt_0p1=('frac_lower_ci_lt_0p1', 'mean'),
        expected_mask_fro_ratio_mean=('expected_mask_fro_ratio_mean', 'mean'),
        expected_train_mask_fro_ratio_mean=('expected_train_mask_fro_ratio_mean', 'mean'),
        mask_attenuation_gap=('mask_attenuation_gap', 'mean'),
        train_mask_attenuation_gap=('train_mask_attenuation_gap', 'mean'),
    )
)

plot_manifest = {}
for (depth, architecture), plot_df in ci_mean_df.groupby(['depth', 'architecture'], sort=True):
    plot_manifest[f'ci_summary_depth{depth}_{architecture}'] = multi_metric_panel_by_layer(
        df=plot_df,
        x_col='checkpoint_step',
        y_cols=['mean_lower_ci', 'frac_lower_ci_gt_0p9', 'frac_lower_ci_lt_0p1'],
        titles=['Mean lower CI', 'Fraction lower CI > 0.9', 'Fraction lower CI < 0.1'],
        subdir='ci_mask',
        stem=f'ci_summary_depth{depth}_{architecture}',
        hline_at_one=False,
    )
    plot_manifest[f'attenuation_depth{depth}_{architecture}'] = multi_metric_panel_by_layer(
        df=plot_df,
        x_col='checkpoint_step',
        y_cols=['mask_attenuation_gap', 'train_mask_attenuation_gap', 'expected_mask_fro_ratio_mean'],
        titles=['Mask attenuation gap', 'Train-mask attenuation gap', 'Expected masked Fro ratio'],
        subdir='ci_mask',
        stem=f'attenuation_depth{depth}_{architecture}',
        hline_at_one=False,
    )
len(plot_manifest)


20

In [6]:
final_ci_df = ci_mask_df.sort_values('checkpoint_step').groupby(['run_name', 'layer_name'], as_index=False).tail(1)
final_ci_mean_df = (
    final_ci_df.groupby(['architecture', 'depth', 'layer_name'], as_index=False)
    .agg(
        expected_mask_fro_ratio_mean=('expected_mask_fro_ratio_mean', 'mean'),
        expected_train_mask_fro_ratio_mean=('expected_train_mask_fro_ratio_mean', 'mean'),
        mask_attenuation_gap=('mask_attenuation_gap', 'mean'),
    )
)
for architecture in ['tied', 'untied']:
    arch_df = final_ci_mean_df[final_ci_mean_df['architecture'] == architecture].copy()
    ordered_layers = [layer for layer in LAYER_ORDER if layer in set(arch_df['layer_name'])]
    masked_matrix = arch_df.pivot(index='depth', columns='layer_name', values='expected_mask_fro_ratio_mean').reindex(columns=ordered_layers).sort_index()
    attenuation_matrix = arch_df.pivot(index='depth', columns='layer_name', values='mask_attenuation_gap').reindex(columns=ordered_layers).sort_index()
    plot_manifest[f'masked_heatmap_{architecture}'] = heatmap(
        matrix=masked_matrix.to_numpy(),
        row_labels=[str(idx) for idx in masked_matrix.index],
        col_labels=list(masked_matrix.columns),
        title=f'Final expected masked Fro ratio | {architecture}',
        colorbar_label='Masked Fro ratio',
        subdir='ci_mask',
        stem=f'masked_heatmap_{architecture}',
        vmin=0.4,
        vmax=1.05,
        annotate=True,
    )
    plot_manifest[f'attenuation_heatmap_{architecture}'] = heatmap(
        matrix=attenuation_matrix.to_numpy(),
        row_labels=[str(idx) for idx in attenuation_matrix.index],
        col_labels=list(attenuation_matrix.columns),
        title=f'Final CI attenuation gap | {architecture}',
        colorbar_label='Raw ratio - masked ratio',
        subdir='ci_mask',
        stem=f'attenuation_heatmap_{architecture}',
        vmin=0.0,
        vmax=max(0.05, float(attenuation_matrix.to_numpy().max())),
        cmap='OrRd',
        annotate=True,
    )
len(plot_manifest)


24

In [7]:
representative_runs_df = selected_manifest_df.sort_values(['depth', 'architecture', 'replicate', 'run_name']).groupby(['depth', 'architecture'], as_index=False).first()
requested_steps = [5000, 20000, 40000]

for _, run_row in representative_runs_df.iterrows():
    available_steps = [step for step in requested_steps if step in run_row['checkpoint_steps']]
    if not available_steps:
        continue
    for layer_name in [layer for layer in LAYER_ORDER if layer in set(ci_mask_df[ci_mask_df['run_name'] == run_row['run_name']]['layer_name'])]:
        arrays = []
        titles = []
        for step in available_steps:
            component_model, _target_model, config = load_component_model_for_checkpoint(spd_run_dir=Path(run_row['spd_run_dir']), step=int(step), device=DEVICE)
            probe_batch = make_probe_batch(config=config, device=DEVICE)
            _pre_sigmoid, ci_lower, _ci_upper = collect_ci_outputs(component_model=component_model, batch=probe_batch, sampling=config.sampling)
            arrays.append(ci_lower[layer_name].detach().cpu().numpy())
            titles.append(f'{step // 1000}k')
        plot_manifest[f'ci_hist_{run_row["run_name"]}_{layer_name}'] = histogram_triptych(
            arrays=arrays,
            titles=titles,
            super_title=f'{run_row["run_name"]} | {layer_name} lower-CI histogram',
            subdir='ci_mask',
            stem=f'ci_hist_{run_row["run_name"]}_{layer_name}'.replace('.', '_'),
            bins=20,
        )

save_json(plot_manifest, 'plots/ci_mask/manifest.json')
plot_manifest


{'ci_summary_depth2_tied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/ci_mask/ci_summary_depth2_tied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/ci_mask/ci_summary_depth2_tied.pdf'},
 'attenuation_depth2_tied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/ci_mask/attenuation_depth2_tied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/ci_mask/attenuation_depth2_tied.pdf'},
 'ci_summary_depth2_untied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/ci_mask/ci_summary_depth2_untied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/ci_mask/ci_summary_depth2_untied.pdf'},
 'attenuation_depth2_untied': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/ci_mask/attenuation_depth2_untied.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/ci_mask/attenuation_depth2_untied.p